# Deploying AI
## Assignment 1: Evaluating Summaries

A key application of LLMs is to summarize documents. In this assignment, we will not only summarize documents, but also evaluate the quality of the summary and return the results using structured outputs.

**Instructions:** please complete the sections below stating any relevant decisions that you have made and showing the code substantiating your solution.

## Select a Document

Please select one out of the following articles:

+ [Managing Oneself, by Peter Druker](https://www.thecompleteleader.org/sites/default/files/imce/Managing%20Oneself_Drucker_HBR.pdf)  (PDF)
+ [The GenAI Divide: State of AI in Business 2025](https://www.artificialintelligence-news.com/wp-content/uploads/2025/08/ai_report_2025.pdf) (PDF)
+ [What is Noise?, by Alex Ross](https://www.newyorker.com/magazine/2024/04/22/what-is-noise) (Web)

# Load Secrets

In [1]:
import os
import sys
sys.path.append(os.path.abspath('../05_src'))
%load_ext dotenv
%dotenv ../05_src/.env
%dotenv ../05_src/.secrets

In [2]:
from utils.logger import get_logger
_logs = get_logger(__name__)

In [3]:
from openai import OpenAI
import os
client = OpenAI(base_url='https://k7uffyg03f.execute-api.us-east-1.amazonaws.com/prod/openai/v1', 
                api_key='any value',
                default_headers={"x-api-key": os.getenv('API_GATEWAY_KEY')})

In [4]:
response = client.responses.create(
    model = 'gpt-4o-mini',
    input = 'Hello world!'
    
)

print(response.output_text)

Hello! How can I assist you today?


## Load Document

Depending on your choice, you can consult the appropriate set of functions below. Make sure that you understand the content that is extracted and if you need to perform any additional operations (like joining page content).

### PDF

You can load a PDF by following the instructions in [LangChain's documentation](https://docs.langchain.com/oss/python/langchain/knowledge-base#loading-documents). Notice that the output of the loading procedure is a collection of pages. You can join the pages by using the code below.

```python
document_text = ""
for page in docs:
    document_text += page.page_content + "\n"
```

### Web

LangChain also provides a set of web loaders, including the [WebBaseLoader](https://docs.langchain.com/oss/python/integrations/document_loaders/web_base). You can use this function to load web pages.

In [5]:
import requests
file_url = 'https://www.thecompleteleader.org/sites/default/files/imce/Managing%20Oneself_Drucker_HBR.pdf'
book = requests.get(file_url)
book

<Response [200]>

In [6]:
dict(book.headers)

{'Date': 'Sun, 03 May 2026 21:56:35 GMT',
 'Server': 'Apache',
 'X-Content-Type-Options': 'nosniff',
 'Upgrade': 'h2',
 'Connection': 'Upgrade, Keep-Alive',
 'Last-Modified': 'Sun, 22 Aug 2021 17:13:44 GMT',
 'ETag': '"2d611-5ca2905de224c"',
 'Accept-Ranges': 'bytes',
 'Content-Length': '185873',
 'Cache-Control': 'max-age=31536000',
 'Expires': 'Mon, 03 May 2027 21:56:35 GMT',
 'Vary': 'User-Agent',
 'Keep-Alive': 'timeout=5, max=100',
 'Content-Type': 'application/pdf'}

## Generation Task

Using the OpenAI SDK, please create a **structured outut** with the following specifications:

+ Use a model that is NOT in the GPT-5 family.
+ Output should be a Pydantic BaseModel object. The fields of the object should be:

    - Author
    - Title
    - Relevance: a statement, no longer than one paragraph, that explains why is this article relevant for an AI professional in their professional development.
    - Summary: a concise and succinct summary no longer than 1000 tokens.
    - Tone: the tone used to produce the summary (see below).
    - InputTokens: number of input tokens (obtain this from the response object).
    - OutputTokens: number of tokens in output (obtain this from the response object).
       
+ The summary should be written using a specific and distinguishable tone, for example,  "Victorian English", "African-American Vernacular English", "Formal Academic Writing", "Bureaucratese" ([the obscure language of beaurocrats](https://tumblr.austinkleon.com/post/4836251885)), "Legalese" (legal language), or any other distinguishable style of your preference. Make sure that the style is something you can identify. 
+ In your implementation please make sure to use the following:

    - Instructions and context should be stored separately and the context should be added dynamically. Do not hard-code your prompt, instead use formatted strings or an equivalent technique.
    - Use the developer (instructions) prompt and the user prompt.


In [7]:
system_prompt = "You are a specialist in summarizing texts for busy professionals who speaks like Gen Z."

In [8]:
import requests
from langchain_community.document_loaders import PyPDFLoader
import tempfile

url = "https://www.thecompleteleader.org/sites/default/files/imce/Managing%20Oneself_Drucker_HBR.pdf"

pdf_bytes = requests.get(url).content

with tempfile.NamedTemporaryFile(delete=False, suffix=".pdf") as tmp:
    tmp.write(pdf_bytes)
    pdf_path = tmp.name

loader = PyPDFLoader(pdf_path)
pages = loader.load()


In [9]:
#joining pages
book_text = "\n\n".join([p.page_content for p in pages])

In [10]:
#Because the book is too long, I'm parsing it into chunks
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size=8000,      # safe for GPT‑4o
    chunk_overlap=500
)

chunks = splitter.split_text(book_text)
len(chunks)


10

In [11]:
# after chunking, I am using the llm to summarize each chunk to generate multiple short summaries. After this, I will merge each summary to produce a big summary.
summaries = []

for chunk in chunks:
    resp = client.responses.create(
        model="gpt-4o-mini",
        input=f"Summarize this text in 1000 tokens:\n\n{chunk}"
    )
    text = resp.output[0].content[0].text
    summaries.append(text)

In [12]:
summaries

['Peter F. Drucker\'s article "Managing Oneself," published in the Harvard Business Review, emphasizes the importance of self-management in today\'s knowledge economy, where individuals must take charge of their own careers rather than relying on organizations to guide them. In a world filled with opportunities, those with ambition and intelligence can excel regardless of their backgrounds, but success also comes with the responsibility of understanding oneself.\n\nTo thrive, individuals need to cultivate a profound self-awareness that includes recognizing their strengths, weaknesses, work styles, values, and environments where they can make the most significant contributions. Drucker argues that operating from a basis of personal strengths and self-knowledge is essential for achieving lasting excellence.\n\nHe suggests several critical questions for individuals to consider as they navigate their careers:\n\n1. **What Are My Strengths?** Understanding one\'s strengths is vital, and Dru

In [13]:
combined_summaries = "\n\n".join(summaries)

In [14]:
# #prompt = f"""
#     You are a specialist in summarizing texts. 
#     Using only the text below, do the following:
    
#     1. Identify the book's title and author.
#     2. Summarize concisely in less than 1000 tokens the main objective of the paper.
#     3. Identify and state the relevance of this book in less than 2 sentences for AI professionals and their development.
#     4. State number of input tokens.
#     5. State number of output tokens.
#     6. State the tone of the book.

        
#     The book is the following: 
#     <book>
#     {summaries}
#     </book>

#     Provide your response in the following format:
#     Title: <title>
#     Author: <author>
#     Summary: <summary>
#     Relevance: <relevance>
#     Input Tokens: <input_tokens>
#     Output Tokens: <output_tokens>
#     Tone: <tone>

# #"""#

In [15]:
prompt = f"""
You are a specialist in summarizing texts.

Using ONLY the text below, extract the following information:

1. The book's title and author.
2. A concise summary of the book's main objective (no more than 1000 tokens).
3. A short explanation of why this book is relevant for AI professionals.
4. The tone of the book.

Do NOT fabricate information not present in the text.

The book text is:

<book>
{combined_summaries}
</book>
"""


In [16]:
prompt

'\nYou are a specialist in summarizing texts.\n\nUsing ONLY the text below, extract the following information:\n\n1. The book\'s title and author.\n2. A concise summary of the book\'s main objective (no more than 1000 tokens).\n3. A short explanation of why this book is relevant for AI professionals.\n4. The tone of the book.\n\nDo NOT fabricate information not present in the text.\n\nThe book text is:\n\n<book>\nPeter F. Drucker\'s article "Managing Oneself," published in the Harvard Business Review, emphasizes the importance of self-management in today\'s knowledge economy, where individuals must take charge of their own careers rather than relying on organizations to guide them. In a world filled with opportunities, those with ambition and intelligence can excel regardless of their backgrounds, but success also comes with the responsibility of understanding oneself.\n\nTo thrive, individuals need to cultivate a profound self-awareness that includes recognizing their strengths, weakn

In [17]:
response = client.responses.create(
    model = 'gpt-4o-mini',
    #model = 'gpt-4o-mini', # depending on the tier we have available, we might need to update the model to be used
    input = prompt
)

Trying output with pydantic model. Using <summaries> generated previously as story in prompt.

In [18]:
from langchain.chat_models import init_chat_model

llm = init_chat_model("gpt-4o-mini", 
                      model_provider="openai",
                      base_url='https://k7uffyg03f.execute-api.us-east-1.amazonaws.com/prod/openai/v1',
                      default_headers={"x-api-key": os.getenv('API_GATEWAY_KEY')},
                      )

In [19]:
from typing import Optional
from pydantic import BaseModel

class ArticleSummary(BaseModel):
    Author: str
    Title: str
    Relevance: str
    Summary: str
    Tone: str
    InputTokens: int
    OutputTokens: int

structured_llm = llm.with_structured_output(ArticleSummary)
result = structured_llm.invoke(prompt)

#jk = structured_llm.invoke("Tell me a joke about cats")

Failed to multipart ingest runs: langsmith.utils.LangSmithError: Failed to POST https://api.smith.langchain.com/runs/multipart in LangSmith API. HTTPError('403 Client Error: Forbidden for url: https://api.smith.langchain.com/runs/multipart', '{"error":"Forbidden"}\n')


Failed to send compressed multipart ingest: langsmith.utils.LangSmithError: Failed to POST https://api.smith.langchain.com/runs/multipart in LangSmith API. HTTPError('403 Client Error: Forbidden for url: https://api.smith.langchain.com/runs/multipart', '{"error":"Forbidden"}\n')


In [20]:
result

ArticleSummary(Author='Peter F. Drucker', Title='Managing Oneself', Relevance="This book is highly relevant for AI professionals as it emphasizes the importance of self-awareness and understanding one's strengths in a rapidly evolving technological landscape. As AI professionals often face changing roles and environments, the principles of self-management can help them adapt quickly and leverage their unique skills effectively, ensuring continuous personal and professional growth.", Summary='"Managing Oneself" underlines the importance of self-management in the knowledge economy, where individuals are encouraged to take control of their careers instead of relying solely on organizations. The book advocates for developing profound self-awareness, focusing on one\'s strengths, weaknesses, work styles, and values, to navigate personal and professional growth effectively. Drucker suggests critical self-reflective questions, such as identifying strengths through feedback analysis, recognizi

In [ ]:
# final_summary = response.output_text

In [ ]:
# from IPython.display import display, Markdown

# display(Markdown(response.output_text))

# Evaluate the Summary

Use the DeepEval library to evaluate the **summary** as follows:

+ Summarization Metric:

    - Use the [Summarization metric](https://deepeval.com/docs/metrics-summarization) with a **bespoke** set of assessment questions.
    - Please use, at least, five assessment questions.

+ G-Eval metrics:

    - In addition to the standard summarization metric above, please implement three evaluation metrics: 
    
        - [Coherence or clarity](https://deepeval.com/docs/metrics-llm-evals#coherence)
        - [Tonality](https://deepeval.com/docs/metrics-llm-evals#tonality)
        - [Safety](https://deepeval.com/docs/metrics-llm-evals#safety)

    - For each one of the metrics above, implement five assessment questions.

+ The output should be structured and contain one key-value pair to report the score and another pair to report the explanation:

    - SummarizationScore
    - SummarizationReason
    - CoherenceScore
    - CoherenceReason
    - ...

In [21]:
#input
input = book_text
#output
actual_output = combined_summaries

In [22]:
from deepeval import evaluate
from deepeval.metrics import AnswerRelevancyMetric
from deepeval.test_case import LLMTestCase
from deepeval.models import GPTModel

model = GPTModel(
    model="gpt-4o-mini",
    temperature=0,
    # api_key='any value',
    _openai_api_key='any value',
    default_headers={"x-api-key": os.getenv('API_GATEWAY_KEY')},
    base_url='https://k7uffyg03f.execute-api.us-east-1.amazonaws.com/prod/openai/v1',
)

In [23]:
from deepeval import evaluate
from deepeval.test_case import LLMTestCase
from deepeval.metrics import SummarizationMetric

metric = SummarizationMetric(
    threshold=0.5,
    assessment_questions=[
        "Does the summary capture the main idea of the text?",
        "Does the summary match the text factually?"
        #"Is the summary easy to understand?",
        #"Is the summary free of unnecessary details?",
        #"Is the summary punctuated correctly?"

    ],
    model=model
)

# To run metric as a standalone
# metric.measure(test_case)
# print(metric.score, metric.reason)

test_case = LLMTestCase(
    input=prompt.format(story=book_text),
    actual_output=response.output_text,
)

evaluate(test_cases=[test_case], metrics=[metric])

results = evaluate(
    test_cases=[test_case],
    metrics=[metric]
)

Summarization_FEEDBACK = metric.reason

✨ You're running DeepEval's latest Summarization Metric! (using gpt-4o-mini, strict=False, async_mode=True)...

Output()



Metrics Summary

  - ✅ Summarization (score: 0.8125, threshold: 0.5, strict: False, evaluation model: gpt-4o-mini, reason: The score is 0.81 because the summary includes some contradictions and extra information not found in the original text, which affects its accuracy. However, it still captures the main ideas effectively, leading to a relatively high score., error: None)

For test case:

  - input: 
You are a specialist in summarizing texts.

Using ONLY the text below, extract the following information:

1. The book's title and author.
2. A concise summary of the book's main objective (no more than 1000 tokens).
3. A short explanation of why this book is relevant for AI professionals.
4. The tone of the book.

Do NOT fabricate information not present in the text.

The book text is:

<book>
Peter F. Drucker's article "Managing Oneself," published in the Harvard Business Review, emphasizes the importance of self-management in today's knowledge economy, where individuals must take ch

✓ Evaluation completed 🎉! (time taken: 34.58s | token cost: 0.00256485 USD)
» Test Results (1 total tests):
   » Pass Rate: 100.0% | Passed: 1 | Failed: 0

 ================================================================================ 

» What to share evals with your team, or a place for your test cases to live? ❤️ 🏡
  » Run 'deepeval view' to analyze and save testing results on Confident AI.

✨ You're running DeepEval's latest Summarization Metric! (using gpt-4o-mini, strict=False, async_mode=True)...

Output()



Metrics Summary

  - ✅ Summarization (score: 0.8125, threshold: 0.5, strict: False, evaluation model: gpt-4o-mini, reason: The score is 0.81 because the summary includes contradictions regarding the emphasis on managing relationships and unique performance styles, which are not present in the original text. Additionally, it introduces extra information about AI professionals and interdisciplinary skills that were not mentioned in the original text. Despite these issues, the summary captures some key themes, which is commendable., error: None)

For test case:

  - input: 
You are a specialist in summarizing texts.

Using ONLY the text below, extract the following information:

1. The book's title and author.
2. A concise summary of the book's main objective (no more than 1000 tokens).
3. A short explanation of why this book is relevant for AI professionals.
4. The tone of the book.

Do NOT fabricate information not present in the text.

The book text is:

<book>
Peter F. Drucker's art

✓ Evaluation completed 🎉! (time taken: 31.6s | token cost: 0.0025844999999999996 USD)
» Test Results (1 total tests):
   » Pass Rate: 100.0% | Passed: 1 | Failed: 0

 ================================================================================ 

» What to share evals with your team, or a place for your test cases to live? ❤️ 🏡
  » Run 'deepeval view' to analyze and save testing results on Confident AI.

In [24]:
metric.measure(test_case)
from IPython.display import display, Markdown
display(Markdown(f'**Score**: {metric.score}'))
display(Markdown(f'**Reason**: {metric.reason}'))

Output()

**Score**: 0.8

**Reason**: The score is 0.80 because while the summary captures the essence of the original text, it introduces extra information that is not present in the original, which could lead to misunderstandings about the content. However, there are no contradictions, and the overall coherence remains strong.

G-EVAL Metrics

In [25]:
from deepeval.metrics import GEval
from deepeval.test_case import LLMTestCaseParams

# METRIC 1: TONALITY
clarity = GEval(
    name="Clarity",
    criteria=(
        "Evaluate whether the response uses clear and direct language."
        "Check if the explanation avoids jargon or explains it when used."
        #"Assess whether complex ideas are presented in a way that's easy to follow.",
        #"Identify any vague or confusing parts that reduce understanding."
),
    evaluation_params=[LLMTestCaseParams.ACTUAL_OUTPUT],
    model=model
)

#METRIC 2: TONALITY (PROFESSIONALISM)
professionalism = GEval(
    name="Professionalism",
    criteria=(
        "Determine whether the actual output maintains a professional tone throughout."
        "Evaluate if the language in the actual output reflects expertise and domain-appropriate formality."
        #"Ensure the actual output stays contextually appropriate and avoids casual or ambiguous expressions.",
        #"Check if the actual output is clear, respectful, and avoids slang or overly informal phrasing."
    ),
    evaluation_params=[LLMTestCaseParams.ACTUAL_OUTPUT],
    model=model
)

#METRIC 3: SAFETY
pii_leakage = GEval(
    name="pii_Leakage",
    criteria=(
        #"Check whether the output includes any real or plausible personal information (e.g., names, phone numbers, emails)."
        "Make sure the summary free of any ideas or text that is not part of the original story."
        #"Identify any hallucinated PII or training data artifacts that could compromise user privacy."
        #"Ensure the output uses placeholders or anonymized data when applicable."
        #"Verify that sensitive information is not exposed even in edge cases or unclear prompts."
    ),
    evaluation_params=[LLMTestCaseParams.ACTUAL_OUTPUT],
)

In [ ]:
test_case = LLMTestCase(
    input=prompt.format(story=book_text),
    actual_output=response.output_text
    )
evaluate(test_cases=[test_case], metrics=[clarity])

In [ ]:
evaluate(test_cases=[test_case], metrics=[professionalism])

In [ ]:
evaluate(test_cases=[test_case], metrics=[pii_leakage])

# Enhancement

Of course, evaluation is important, but we want our system to self-correct.  

+ Use the context, summary, and evaluation that you produced in the steps above to create a new prompt that enhances the summary.
+ Evaluate the new summary using the same function.
+ Report your results. Did you get a better output? Why? Do you think these controls are enough?

In [36]:
enhanced_prompt = f"""

You are an assistant that helps rewrite summaries based on evaluation feedback.

Using the following information:

1. Original context:
{input}

2. First summary:
{actual_output}

3. Evaluation feedback:
{Summarization_FEEDBACK}

Rewrite the summary for the original context so that it:
- fixes all issues mentioned in the evaluation feedback
- improves coverage of key points
- removes hallucinations
- increases clarity and coherence
- stays concise
- preserves factual accuracy
"""

In [37]:
from openai import OpenAI
client = OpenAI(base_url='https://k7uffyg03f.execute-api.us-east-1.amazonaws.com/prod/openai/v1', 
                api_key='any value',
                default_headers={"x-api-key": os.getenv('API_GATEWAY_KEY')})
response = client.responses.create(
    model = 'gpt-4o-mini',
    #model = 'gpt-4o-mini', # depending on the tier we have available, we might need to update the model to be used
    input = enhanced_prompt
)

IMPROVED_SUMMARY = response.output[0].content[0].text
print(IMPROVED_SUMMARY)

**Managing Oneself: A Summary**

In "Managing Oneself," Peter F. Drucker explores the vital role of self-management in today’s knowledge economy. As traditional companies no longer guide career paths, individuals must take ownership of their careers by understanding their strengths, weaknesses, values, and how they work best. Drucker argues that personal self-awareness is essential for achieving lasting success.

Key questions for self-discovery include:

1. **What Are My Strengths?** Drucker emphasizes using feedback analysis to identify one’s strengths. By documenting expected outcomes from decisions and later comparing them to actual results, individuals can discern where they excel and where they need improvement.

2. **How Do I Work?** Recognizing one’s preferred work style—whether learning best through reading or listening, and whether thriving in teams or as a solo contributor—is critical for optimizing performance.

3. **What Are My Values?** Personal ethics and responsibilitie

In [38]:
from deepeval import evaluate

from deepeval.test_case import LLMTestCaseParams

test_case_2 = LLMTestCase(
    input=input,
    actual_output=IMPROVED_SUMMARY,
    retrieval_context=[input],
    context=[input],
)

results_2 = evaluate(
    test_cases=[test_case_2],
    metrics=[metric]   # my SummarizationMetric
)

print(results_2)

✨ You're running DeepEval's latest Summarization Metric! (using gpt-4o-mini, strict=False, async_mode=True)...

Output()



Metrics Summary

  - ✅ Summarization (score: 0.8666666666666667, threshold: 0.5, strict: False, evaluation model: gpt-4o-mini, reason: The score is 0.87 because the summary includes some contradictions regarding personal ethics and responsibilities that were not present in the original text, as well as extra information about historical figures that was not mentioned. However, the overall coherence and relevance of the summary to the main themes of the original text are still strong., error: None)

For test case:

  - input: www.hbr.org
B
 
EST  
 
OF  HBR 1999
 
Managing Oneself
 
by Peter F . Drucker
 
•
 
Included with this full-text 
 
Harvard Business Review
 
 article:
The Idea in Brief—the core idea
The Idea in Practice—putting the idea to work
 
1
 
Article Summary
 
2
 
Managing Oneself
A list of related materials, with annotations to guide further
exploration of the article’s ideas and applications
 
12
 
Further Reading
Success in the knowledge 
economy comes to those who 

✓ Evaluation completed 🎉! (time taken: 21.08s | token cost: 0.004707449999999999 USD)
» Test Results (1 total tests):
   » Pass Rate: 100.0% | Passed: 1 | Failed: 0

 ================================================================================ 

» What to share evals with your team, or a place for your test cases to live? ❤️ 🏡
  » Run 'deepeval view' to analyze and save testing results on Confident AI.

test_results=[TestResult(name='test_case_0', success=True, metrics_data=[MetricData(name='Summarization', threshold=0.5, success=True, score=0.8666666666666667, reason='The score is 0.87 because the summary includes some contradictions regarding personal ethics and responsibilities that were not present in the original text, as well as extra information about historical figures that was not mentioned. However, the overall coherence and relevance of the summary to the main themes of the original text are still strong.', strict_mode=False, evaluation_model='gpt-4o-mini', error=None, evaluation_cost=0.004707449999999999, verbose_logs='Truths (limit=None):\n[\n    "Success in the knowledge economy comes to those who know themselves, including their strengths, values, and how they perform.",\n    "Knowledge workers must manage their own careers and be their own chief executive officers.",\n    "To achieve true excellence, individuals must operate from a combination of their strengths and se

Please, do not forget to add your comments.

In [ ]:
# MY COMMENTS

# The summarization score of my original summary was 0.8. 
# The summarization of my enhanced summarry was 0.867.
# Using the enhanced prompt and the reason from the 'summarization' metric alone, the summary
# did improve and the result was that some of the extra information that the model had added on 
# its own the first time round, did not happen in the improved_summary.
# In the future, I would incorporate the 'score' and 'reason' from all the other G-Eval metrics and include them in the enhanced prompt
# so that the response can be better.


# Submission Information

🚨 **Please review our [Assignment Submission Guide](https://github.com/UofT-DSI/onboarding/blob/main/onboarding_documents/submissions.md)** 🚨 for detailed instructions on how to format, branch, and submit your work. Following these guidelines is crucial for your submissions to be evaluated correctly.

## Submission Parameters

- The Submission Due Date is indicated in the [readme](../README.md#schedule) file.
- The branch name for your repo should be: assignment-1
- What to submit for this assignment:
    + This Jupyter Notebook (assignment_1.ipynb) should be populated and should be the only change in your pull request.
- What the pull request link should look like for this assignment: `https://github.com/<your_github_username>/production/pull/<pr_id>`
    + Open a private window in your browser. Copy and paste the link to your pull request into the address bar. Make sure you can see your pull request properly. This helps the technical facilitator and learning support staff review your submission easily.

## Checklist

+ Created a branch with the correct naming convention.
+ Ensured that the repository is public.
+ Reviewed the PR description guidelines and adhered to them.
+ Verify that the link is accessible in a private browser window.

If you encounter any difficulties or have questions, please don't hesitate to reach out to our team via our Slack. Our Technical Facilitators and Learning Support staff are here to help you navigate any challenges.
